# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I chose **Random Forest** for the Content Refresh Prioritization lane.

The goal is to prioritize content pages that are likely to need a refresh. Random Forest is suitable because it can combine multiple content-performance signals and capture non-linear relationships. I will not judge the model only by its complexity. I will compare it against my Week-4 baseline using the same data, split, and evaluation metric.

The purpose of this experiment is to determine whether the model provides a useful improvement over the existing baseline for the refresh-prioritization decision.

In [11]:
print("All columns:")
for i, col in enumerate(df.columns, 1):
    print(i, col)

All columns:
1 content_id
2 client_id
3 search_volume
4 competition
5 competition_level
6 cpc
7 content_type
8 main_intent
9 word_count
10 char_count
11 provider_used
12 model_used
13 impressions_90d
14 clicks_90d
15 pageviews_90d
16 sessions_90d
17 users_90d
18 engaged_sessions_90d
19 ai_sessions_90d
20 scroll_events_90d
21 days_with_impressions
22 days_with_sessions
23 impressions_last_30d
24 clicks_last_30d
25 sessions_last_30d
26 impressions_prev_30d
27 clicks_prev_30d
28 sessions_prev_30d
29 content_age_days
30 age_tier
31 age_tier_order
32 days_since_last_update
33 freshness_tier
34 word_count_tier
35 char_count_tier
36 ctr
37 avg_position
38 engagement_rate
39 scroll_rate
40 ai_traffic_pct
41 impression_tier
42 position_tier
43 trend_direction
44 trend_pct
45 is_declining_label


In [12]:
print("\nMissing values:")
I chose Random Forest for the Content Refresh Prioritization lane.

The goal is to prioritize content pages that are likely to need a refresh. Random Forest is suitable because it can combine multiple content-performance signals and capture non-linear relationships. I will not judge the model only by its complexity. I will compare it against my Week-4 baseline using the same data, split, and evaluation metric.$0
print(df.isna().sum().sort_values(ascending=False).head(15))


Missing values:
provider_used        21438
word_count            7699
char_count            7699
char_count_tier       7699
word_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
cpc                   2468
search_volume         2468
competition           2468
main_intent           2374
scroll_rate            125
impressions_90d          0
content_type             0
dtype: int64


In [13]:
print("\nTrend direction values:")
print(df["trend_direction"].value_counts(dropna=False))


Trend direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a **client-level holdout** rather than randomly splitting individual rows.

Approximately 20% of clients will be held out for testing, while the remaining clients will be used for training. This prevents pages from the same client appearing in both training and testing.

This is a more honest evaluation for my lane because the model should be tested on clients it did not see during training. I will evaluate the model and Week-4 baseline on the same held-out clients using the same Precision@K metric, so the comparison is fair.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

DATA_PATH = "/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Dataset shape:", df.shape)
print("Declining rate:", round(df["is_declining_label"].mean(), 3))

clients = df["client_id"].dropna().unique()

train_clients, test_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_df = df[df["client_id"].isin(train_clients)].copy()
test_df = df[df["client_id"].isin(test_clients)].copy()

print("\nTraining rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

numeric_candidates = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "days_since_last_update"
]

categorical_candidates = [
    "content_type",
    "main_intent",
    "competition_level",
    "impression_tier",
    "position_tier"
]

numeric_features = [
    col for col in numeric_candidates
    if col in df.columns
]

categorical_features = [
    col for col in categorical_candidates
    if col in df.columns
]

print("\nNumeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nExcluded leakage columns:")
print(["trend_direction", "trend_pct"])

X_train = train_df[numeric_features + categorical_features]
y_train = train_df["is_declining_label"]

X_test = test_df[numeric_features + categorical_features]
y_test = test_df["is_declining_label"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

pipeline.fit(X_train, y_train)

model_probabilities = pipeline.predict_proba(X_test)[:, 1]

model_predictions = (
    model_probabilities >= 0.5
).astype(int)

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(scores)[::-1]
    top_k = order[:k]

    return y_true[top_k].mean()

model_p20 = precision_at_k(
    y_test.values,
    model_probabilities,
    20
)

model_p50 = precision_at_k(
    y_test.values,
    model_probabilities,
    50
)

model_p100 = precision_at_k(
    y_test.values,
    model_probabilities,
    100
)

baseline = test_df.copy()

baseline["baseline_score"] = 0

baseline.loc[
    baseline["days_since_last_update"] > 365,
    "baseline_score"
] += 30

baseline.loc[
    baseline["ctr"] < 0.10,
    "baseline_score"
] += 25

baseline.loc[
    baseline["avg_position"] > 20,
    "baseline_score"
] += 20

baseline_p20 = precision_at_k(
    baseline["is_declining_label"].values,
    baseline["baseline_score"].values,
    20
)

baseline_p50 = precision_at_k(
    baseline["is_declining_label"].values,
    baseline["baseline_score"].values,
    50
)

baseline_p100 = precision_at_k(
    baseline["is_declining_label"].values,
    baseline["baseline_score"].values,
    100
)

model_auc = roc_auc_score(
    y_test,
    model_probabilities
)

baseline_auc = roc_auc_score(
    baseline["is_declining_label"],
    baseline["baseline_score"]
)

comparison_df = pd.DataFrame([
    {
        "Method": "Week-4 Leakage-Safe Baseline",
        "Precision@20": baseline_p20,
        "Precision@50": baseline_p50,
        "Precision@100": baseline_p100,
        "Average Precision@K": np.mean([
            baseline_p20,
            baseline_p50,
            baseline_p100
        ]),
        "ROC AUC": baseline_auc
    },
    {
        "Method": "Random Forest",
        "Precision@20": model_p20,
        "Precision@50": model_p50,
        "Precision@100": model_p100,
        "Average Precision@K": np.mean([
            model_p20,
            model_p50,
            model_p100
        ]),
        "ROC AUC": model_auc
    }
])

print("\nModel vs baseline:")
display(
    comparison_df.round(3)
)

print("\nRandom Forest classification metrics:")

print(
    "Accuracy:",
    round(
        accuracy_score(
            y_test,
            model_predictions
        ),
        3
    )
)

print(
    "Precision:",
    round(
        precision_score(
            y_test,
            model_predictions,
            zero_division=0
        ),
        3
    )
)

print(
    "Recall:",
    round(
        recall_score(
            y_test,
            model_predictions,
            zero_division=0
        ),
        3
    )
)

print(
    "F1:",
    round(
        f1_score(
            y_test,
            model_predictions,
            zero_division=0
        ),
        3
    )
)

print(
    "ROC AUC:",
    round(
        model_auc,
        3
    )
)

feature_names = pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

feature_importances = pipeline.named_steps[
    "model"
].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importances
}).sort_values(
    "importance",
    ascending=False
)

print("\nTop 15 model features:")
display(
    importance_df.head(15)
)

print("\nTop 20 model-ranked pages:")

model_ranked = test_df[
    [
        "content_id",
        "client_id",
        "is_declining_label"
    ]
].copy()

model_ranked["model_score"] = model_probabilities

display(
    model_ranked.sort_values(
        "model_score",
        ascending=False
    ).head(20)
)

print("\nTop 20 baseline-ranked pages:")

baseline_ranked = baseline[
    [
        "content_id",
        "client_id",
        "baseline_score",
        "is_declining_label"
    ]
].copy()

display(
    baseline_ranked.sort_values(
        "baseline_score",
        ascending=False
    ).head(20)
)

Dataset shape: (30000, 45)
Declining rate: 0.542

Training rows: 26581
Test rows: 3419
Training clients: 25
Test clients: 7

Numeric features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'days_since_last_update']

Categorical features:
['content_type', 'main_intent', 'competition_level', 'impression_tier', 'position_tier']

Excluded leakage columns:
['trend_direction', 'trend_pct']

Model vs baseline:


,Method,Precision@20,Precision@50,Precision@100,Average Precision@K,ROC AUC
0,Week-4 Leakage-Safe Baseline,0.50,0.48,0.52,0.500,0.489
1,Random Forest,0.75,0.72,0.68,0.717,0.637



Random Forest classification metrics:
Accuracy: 0.6
Precision: 0.608
Recall: 0.663
F1: 0.634
ROC AUC: 0.637

Top 15 model features:


,feature,importance
5,num__impressions_90d,0.133518
10,num__avg_position,0.121497
4,num__char_count,0.083870
3,num__word_count,0.083859
7,num__pageviews_90d,0.064131
8,num__sessions_90d,0.062718
12,num__scroll_rate,0.059084
9,num__ctr,0.058645
6,num__clicks_90d,0.045709
0,num__search_volume,0.043220



Top 20 model-ranked pages:


,content_id,client_id,is_declining_label,model_score
2115,content_a0c3be8b0794,client_9400f1b21c,1,0.953333
6228,content_e988c1699454,client_8527a891e2,1,0.953333
12972,content_d5b833d82e72,client_9400f1b21c,1,0.946667
25439,content_3d8f6737ad9d,client_a88a7902cb,1,0.936667
8990,content_a38c8f61e246,client_9400f1b21c,0,0.933333
29869,content_6752b50ee577,client_bbb965ab0c,1,0.933333
9325,content_67e739541948,client_a88a7902cb,1,0.930000
8484,content_a128777972dd,client_9400f1b21c,1,0.930000
16953,content_201c5c2257b9,client_bbb965ab0c,1,0.930000
20940,content_b5252147f2d0,client_9400f1b21c,1,0.926667



Top 20 baseline-ranked pages:


,content_id,client_id,baseline_score,is_declining_label
44,content_793b7376a0e5,client_8527a891e2,45,1
29988,content_9bb9a0584cae,client_8527a891e2,45,1
13,content_a5a2fbc76336,client_8527a891e2,45,0
29961,content_326a540b3a6e,client_8527a891e2,45,1
29930,content_52a0cf81179f,client_8527a891e2,45,1
20072,content_444cb6541fc2,client_8527a891e2,45,0
20088,content_61f1696827fb,client_a88a7902cb,45,1
20113,content_aa9d3a51bb6d,client_a88a7902cb,45,1
20137,content_ed8d3d6827b7,client_a88a7902cb,45,1
20198,content_282c159379cf,client_a88a7902cb,45,0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

The Random Forest performed better than the leakage-safe Week-4 baseline on all three ranking metrics. Precision@20 increased from 0.50 to 0.75, Precision@50 increased from 0.48 to 0.72, and Precision@100 increased from 0.52 to 0.68. Average Precision@K increased from 0.500 for the baseline to 0.717 for the Random Forest. ROC AUC also increased from 0.489 to 0.637.

The Random Forest was not perfect. In the top-20 model-ranked pages, most pages had an is_declining_label of 1, but some high-scored pages had a label of 0. These are false positives where the model ranked a page highly even though the observed label did not indicate decline.

The most important model features were impressions_90d, avg_position, char_count, word_count, pageviews_90d, sessions_90d, scroll_rate, and ctr. This shows that the model used a combination of search-performance and content-related signals rather than relying on a single feature.

The baseline produced many tied scores, with several pages receiving a baseline score of 45. This makes the baseline ranking less differentiated. The Random Forest produces a probability score for each page, allowing a more differentiated ranking.

The Week-4 heuristic originally included trend_direction, but trend_direction is also used to construct the ML-08 target. Therefore, it was excluded from this comparison to avoid target leakage. The comparison uses the remaining leakage-safe baseline signals.

Overall, the Random Forest provided stronger ranking performance than the leakage-safe baseline on the held-out clients. However, the model should be treated as decision support rather than an automatic refresh decision. Human review is still required before deciding whether a page should actually be refreshed.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.